# V3: Model Creation and Training

## 1.1 Importing the Training and Validation Sets

In [7]:
from src.v2.data import BugFixDataset, collate_batch
from torch.utils.data import DataLoader
import pandas as pd
import torch
import json

with open("data/processed/token_to_id.json", "r") as file:
    token_to_id = json.load(file)

with open("data/processed/label_to_id.json", "r") as file:
    label_to_id = json.load(file)

train_v2_label_id = pd.read_parquet('data/processed/train_v2_label_id.parquet')
validation_v2_label_id = pd.read_parquet('data/processed/validation_v2_label_id.parquet')

train_dataset = BugFixDataset(train_v2_label_id)
validation_dataset = BugFixDataset(validation_v2_label_id)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, collate_fn=collate_batch) #returns an iterable a batch that has an iterable of (tokens, labels, mask)
validation_loader = DataLoader(validation_dataset, batch_size=32, shuffle=False, collate_fn=collate_batch)

In [8]:
train_tokens, train_labels, train_mask = next(iter(train_loader))
validation_tokens, validation_labels, validation_mask = next(iter(validation_loader))

assert train_tokens.shape == train_mask.shape
assert validation_tokens.shape == validation_mask.shape

assert train_tokens.shape[0] == train_labels.shape[0]
assert validation_tokens.shape[0] == validation_labels.shape[0]

assert train_tokens.dtype == torch.long
assert validation_tokens.dtype == torch.long

assert train_labels.dtype == torch.long
assert validation_labels.dtype == torch.long

assert train_mask.dtype == torch.bool
assert validation_mask.dtype == torch.bool

print(f'Train Tokens Shape: {train_tokens.shape}')
print(f'Train Labels Shape: {train_labels.shape}')
print(f'Train Mask Shape: {train_mask.shape}')

print(f'Validation Token Shape:{validation_tokens.shape}')
print(f'Validation Labels Shape: {validation_labels.shape}')
print(f'Validation Mask Shape: {validation_mask.shape}')

Train Tokens Shape: torch.Size([32, 194])
Train Labels Shape: torch.Size([32])
Train Mask Shape: torch.Size([32, 194])
Validation Token Shape:torch.Size([32, 76])
Validation Labels Shape: torch.Size([32])
Validation Mask Shape: torch.Size([32, 76])


In [9]:
train_max_length = train_v2_label_id['tokens_ids'].map(len).max()
validation_max_length = validation_v2_label_id['tokens_ids'].map(len).max()

print(f'Train maximum: {train_max_length}')
print(f'Validation maximum: {validation_max_length}')

Train maximum: 554
Validation maximum: 278


## 1.2 Verifying the SinusoidalPositionalEncoding Class

In [10]:
from src.v3.model import SinusoidalPositionalEncoding

positional_encoder = SinusoidalPositionalEncoding(embedding_dim=128, max_length=1024)

test_tensor = torch.zeros(2, 20, 128)
output_tensor = positional_encoder(test_tensor)

assert test_tensor.shape == output_tensor.shape
assert 'positional_encoding' in dict(positional_encoder.named_buffers())
assert not positional_encoder.positional_encoding.requires_grad
assert not torch.equal(output_tensor[0,0], output_tensor[0,1])
assert torch.equal(output_tensor[0,0], output_tensor[1,0])

## 1.3 Testing the TransformerBugFixClassifier Class

In [11]:
from src.v3.model import TransformerBugFixClassifier

model = TransformerBugFixClassifier(vocab_size=len(token_to_id), embedding_dim=128, num_heads=4, num_layers=2, feedforward_dim=256, num_classes=len(label_to_id), max_length=1024, padding_index=token_to_id['<PAD>'],dropout=.1)
model.eval()

with torch.no_grad():
    train_logits = model(train_tokens, train_mask)
    validation_logits = model(validation_tokens, validation_mask)

assert train_logits.shape == torch.Size([32,5])
assert validation_logits.shape == torch.Size([32,5])

assert torch.isfinite(train_logits).all()
assert torch.isfinite(validation_logits).all()

## 1.4 Testing on One Batch

In [12]:
import torch.nn as nn
debug_model = TransformerBugFixClassifier(vocab_size=947, embedding_dim=128, num_heads=4, num_layers=2, feedforward_dim=256, num_classes=5, max_length=1024, padding_index=0,dropout=.1)
train_tokens, train_labels, train_mask = next(iter(train_loader))
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(debug_model.parameters(), lr=1e-3)
debug_model.train()

for i in range(30):
    optimizer.zero_grad()
    logits = debug_model(train_tokens, train_mask)
    loss = criterion(logits, train_labels)
    predictions = logits.argmax(dim=1)

    loss.backward()
    optimizer.step()
    total_loss = loss.item()
    accuracy = (predictions == train_labels).float().mean().item()
    if i in [0, 5, 10, 15, 20, 29]:
        print(f'Step: {i} Loss: {total_loss} Accuracy: {accuracy}')

Step: 0 Loss: 1.744463562965393 Accuracy: 0.0625
Step: 5 Loss: 1.0601807832717896 Accuracy: 0.75
Step: 10 Loss: 0.5611785650253296 Accuracy: 0.875
Step: 15 Loss: 0.19416683912277222 Accuracy: 0.96875
Step: 20 Loss: 0.04905004799365997 Accuracy: 1.0
Step: 29 Loss: 0.009405463933944702 Accuracy: 1.0


## 1.5 Training on the Full Dataset

In [ ]:
from src.v2.train import train_model
torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = TransformerBugFixClassifier(vocab_size=947, embedding_dim=128, num_heads=4, num_layers=2, feedforward_dim=256, num_classes=5, max_length=1024, padding_index=0,dropout=.1).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

history = train_model(model, train_loader=train_loader, validation_loader=validation_loader, criterion=criterion, optimizer=optimizer, epochs=15, checkpoint_path='models/v3_best_model.pt')

Epoch: 1, Train Loss: 0.5033, Train Accuracy: 0.7865 Validation Loss: 0.3830 Validation Accuracy: 0.8390
Epoch: 2, Train Loss: 0.3858, Train Accuracy: 0.8359 Validation Loss: 0.3763 Validation Accuracy: 0.8473
Epoch: 3, Train Loss: 0.3596, Train Accuracy: 0.8483 Validation Loss: 0.3818 Validation Accuracy: 0.8443
Epoch: 4, Train Loss: 0.3388, Train Accuracy: 0.8555 Validation Loss: 0.3517 Validation Accuracy: 0.8610
Epoch: 5, Train Loss: 0.3208, Train Accuracy: 0.8651 Validation Loss: 0.3808 Validation Accuracy: 0.8443
Epoch: 6, Train Loss: 0.3103, Train Accuracy: 0.8711 Validation Loss: 0.3389 Validation Accuracy: 0.8680
Epoch: 7, Train Loss: 0.3029, Train Accuracy: 0.8750 Validation Loss: 0.3590 Validation Accuracy: 0.8576
Epoch: 8, Train Loss: 0.2944, Train Accuracy: 0.8783 Validation Loss: 0.3270 Validation Accuracy: 0.8725
Epoch: 9, Train Loss: 0.2855, Train Accuracy: 0.8808 Validation Loss: 0.3311 Validation Accuracy: 0.8699
Epoch: 10, Train Loss: 0.2808, Train Accuracy: 0.8828 V

## 1.6 Training Outcome

The checkpoint from epoch 10 was selected because it achieved the lowest validation loss of **0.3144**. Validation accuracy reached **0.8773** at this checkpoint. Later epochs continued improving training performance but did not produce a lower validation loss, suggesting that additional training was no longer consistently improving generalization.